In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import os
import shutil
import re



# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

    
from src import config


In [ ]:


def flatten_test_folder(src_root, dst_root):
    """
    Flattens structure from src_root/tomo_id/slice_id 
    into dst_root/tomo_id_slice_id.
    """
    os.makedirs(dst_root, exist_ok=True)

    for tomo_id in sorted(os.listdir(src_root)):
        tomo_path = os.path.join(src_root, tomo_id)
        if not os.path.isdir(tomo_path):
            continue  # skip non-dirs

        for slice_id in sorted(os.listdir(tomo_path)):
            src_file = os.path.join(tomo_path, slice_id)
            if not os.path.isfile(src_file):
                continue
            # new name: tomoId_sliceId
            new_name = f"{tomo_id}_{slice_id}"
            dst_file = os.path.join(dst_root, new_name)
            shutil.copy2(src_file, dst_file)  # copy with metadata
            print(f"Copied {src_file} to {dst_file}")
            # use shutil.move if you want to move instead of copy

    print(f"Flattened files copied to: {dst_root}")


In [ ]:
src = "/data/horse/ws/kein254g-team_project/test"
dst = "/data/horse/ws/kein254g-team_project/test_flat"
flatten_test_folder(src_root=src, dst_root=dst)


In [ ]:
def flatten_train_folder(src_root, dst_root):
    """
    Flattens structure from src_root/tomo_id/slice_id 
    into dst_root/tomo_id_slice_id.
    """
    os.makedirs(dst_root, exist_ok=True)

    for tomo_id in sorted(os.listdir(src_root)):
        tomo_path = os.path.join(src_root, tomo_id)
        if not os.path.isdir(tomo_path):
            continue  # skip non-dirs

        for slice_id in sorted(os.listdir(tomo_path)):
            src_file = os.path.join(tomo_path, slice_id)
            if not os.path.isfile(src_file):
                continue
            # new name: tomoId_sliceId
            new_name = f"{tomo_id}_{slice_id}"
            dst_file = os.path.join(dst_root, new_name)
            shutil.copy2(src_file, dst_file)  # copy with metadata
            print(f"Copied {src_file} to {dst_file}")
            # use shutil.move if you want to move instead of copy

    print(f"Flattened files copied to: {dst_root}")

In [ ]:
src_train = "/data/horse/ws/kein254g-team_project/train"
dst_train = "/data/horse/ws/kein254g-team_project/train_flat"
flatten_train_folder(src_root=src_train, dst_root=dst_train)

In [ ]:
# do extra preprocessing for angstrom spacing 
"""
What YOLOv8 handles

Resize/letterbox to imgsz, stride alignment.
Convert to float32 and normalize to [0, 1] (assumes 8‑bit).
Basic train-time augs (scale, flip, HSV, mosaic).


What you should handle (outside YOLO)

Voxel spacing: resample XY to a common physical spacing; convert any physical thresholds to pixels.
Z dimension: create MIPs/slabs with a fixed physical thickness (k = round(thickness / z_spacing)).
Bit depth: convert 16‑bit to 8‑bit (clip to percentiles, scale to 0–255) to avoid wrong normalization.
Grayscale: replicate to 3 channels (or customize the model to 1‑channel).
Denoise/contrast: domain‑appropriate filtering (e.g., Gaussian, CLAHE).
Labels: if you resample, scale annotations by the same per‑axis factors.

"""

In [ ]:
df = pd.read_csv("/home/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/raw/train_labels.csv")
df.head()

In [ ]:
src_train = Path('/data/horse/ws/kein254g-team_project/train')
count = 0
for file_path in src_train.rglob('*.jpg'): 
    tomo_id = file_path.parent.name.removeprefix('tomo_') # '2483bb'
    m = re.search(r'slice[_-]?(\d+)', file_path.stem)
    slice_number = int(m.group(1)) # 25
    print(tomo_id)
    print(slice_number)
    print(file_path)
    #shutil.move(str(file_path), str(target))  # robust across mounts
print(count)

In [3]:
src_train = Path('/data/horse/ws/kein254g-team_project/train')


def find_tomo_path (tomoId, sliceNumber):
    for file_path in src_train.rglob('*.jpg'): 
        idTomo = file_path.parent.name.removeprefix('tomo_') # '2483bb'
        m = re.search(r'slice[_-]?(\d+)', file_path.stem)
        slice_number = int(m.group(1))
        tomo_id = "tomo_" + idTomo

        if sliceNumber is None or sliceNumber == -1.0:
            print("Slice number is None or -1")
            return ""


        if (tomo_id == tomoId and sliceNumber == slice_number):
            print("Match slice number: " + str(slice_number))
            print("Match tomo nidumber: " + str(tomo_id))
            print(file_path)
            return str(file_path)
            

    return ""
   
    #shutil.move(str(file_path), str(target))  # robust across mounts

#find_tomo_path("tomo_2483bb", 10)



In [ ]:

df = pd.read_csv('/home/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/raw/train_labels.csv') 
#dfNew = pd.read_csv('/home/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/raw/train_labels_new.csv') 

for _,row in df.iterrows():
    print(row['tomo_id'])
    print(row['Motor axis 0'])
    file_path = find_tomo_path(row['tomo_id'], row['Motor axis 0'])
    d = {'row_id': row['row_id'],
         'tomo_id': row['tomo_id'],
         'Motor axis 0': row['Motor axis 0'],
         'Motor axis 1': row['Motor axis 1'],
         'Motor axis 2': row['Motor axis 2'],
         'Array shape (axis 0)': row['Array shape (axis 0)'],
         'Array shape (axis 1)': row['Array shape (axis 1)'],
         'Array shape (axis 2)': row['Array shape (axis 2)'],
         'Voxel spacing': row['Voxel spacing'],
         'Number of motors': row['Number of motors'],
         'Path': file_path}
    df.insert(10, 'Path', file_path)
    #dfNew = pd.concat([dfNew, pd.DataFrame([d])], ignore_index=True)

    

#dfNew.to_csv('/home/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/raw/train_labels_new.csv', index=False)



tomo_003acc
-1.0
Slice number is None or -1
tomo_00e047
169.0
Match slice number: 169
Match tomo nidumber: tomo_00e047
/data/horse/ws/kein254g-team_project/train/tomo_00e047/slice_0169.jpg


ValueError: cannot insert Path, already exists

In [ ]:
# normalized x,y,w,h , letterbox size, original size






